In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath('../../..'))
from model_utils.plots import plot_results
from GNN.DynamicSimilarities.Node_Embedding.lstm import LSTM
from dataset import TimeSeriesDataset
import time
import itertools
import math
import pickle

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.11.0+cu128
CUDA version: 12.8
CUDA available: True


In [ ]:
import os
import shutil
import stat
import time

def handle_remove_readonly(func, path, exc):
    # Callback to handle read-only files on Windows
    excvalue = exc[1]
    if func in (os.rmdir, os.remove, os.unlink) and excvalue.errno == 13: # EACCES
        os.chmod(path, stat.S_IWRITE)
        func(path)
    else:
        raise

# Clean up directories from previous runs
dirs_to_cleanup = ['best_models', 'grid_search_plots', 'training_logs', 'inference_logs']
for dir_path in dirs_to_cleanup:
    if os.path.exists(dir_path):
        # Retry a few times in case of transient locks
        for i in range(3):
            try:
                shutil.rmtree(dir_path, ignore_errors=False, onerror=handle_remove_readonly)
                print(f"Removed directory: {dir_path}")
                break
            except Exception as e:
                if i < 2:
                    time.sleep(1) # Wait a bit before retrying
                else:
                    print(f"Error removing {dir_path}: {e}")

Removed directory: best_models
Removed directory: grid_search_plots
Removed directory: training_logs
Removed directory: inference_logs


In [ ]:
DATA_PATH = '../dataset/data_andre.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

#NUM_ITEMS = 100
#df = df[df['item_id'].isin(df['item_id'].unique()[:NUM_ITEMS])]

Loading data from ../dataset/data_andre.feather...


In [ ]:
import holidays

# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------


# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------


# ensure day 2022-09-24 is the first day of test set
df = df.sort_values([DATE_COL, 'item_id', 'store_id']).reset_index(drop=True)


# -----------------------------------------------------------------------------
# SORT
# -----------------------------------------------------------------------------
df = df.sort_values([DATE_COL, "item_id", "store_id"]).reset_index(drop=True)

# -----------------------------------------------------------------------------
# BASIC CALENDAR PARTS
# -----------------------------------------------------------------------------
df["day_of_week"]   = df[DATE_COL].dt.dayofweek.astype(int)         # 0=Mon
df["day_of_month"]  = df[DATE_COL].dt.day.astype(int)
df["month"]         = df[DATE_COL].dt.month.astype(int)
df["moy"]           = (df["month"] - 1).astype(int)
df["quarter"]       = df[DATE_COL].dt.quarter.astype(int)
df["doy"]           = (df[DATE_COL].dt.dayofyear - 1).astype(int)
df["week_of_year"]  = df[DATE_COL].dt.isocalendar().week.astype(int)
df["year"]          = df[DATE_COL].dt.year.astype(int)

# weekend
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_monday"] = (df["day_of_week"] == 0).astype(int)
df["is_friday"] = (df["day_of_week"] == 4).astype(int)


# month / quarter boundaries
df["is_month_start"]   = df[DATE_COL].dt.is_month_start.astype(int)
df["is_month_end"]     = df[DATE_COL].dt.is_month_end.astype(int)
df["is_quarter_start"] = df[DATE_COL].dt.is_quarter_start.astype(int)
df["is_quarter_end"]   = df[DATE_COL].dt.is_quarter_end.astype(int)

# optional: week of month
df["week_of_month"] = ((df["day_of_month"] - 1) // 7 + 1).astype(int)

# -----------------------------------------------------------------------------
# CYCLICAL ENCODINGS
# -----------------------------------------------------------------------------
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

df["doy_sin"] = np.sin(2 * np.pi * df["doy"] / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df["doy"] / 365.25)

# -----------------------------------------------------------------------------
# US HOLIDAYS
# -----------------------------------------------------------------------------
us_holidays = holidays.US()
holiday_dates = pd.to_datetime(sorted(us_holidays.keys()))

df["is_holiday"] = df[DATE_COL].isin(holiday_dates).astype(int)

# named holidays
df["is_christmas"] = (
    (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 25)
).astype(int)

df["is_thanksgiving"] = df[DATE_COL].apply(
    lambda x: 1 if us_holidays.get(x) == "Thanksgiving Day" else 0
)

# Black Friday (very useful for retail)
thanksgiving_dates = pd.to_datetime(
    [d for d, name in us_holidays.items() if name == "Thanksgiving Day"]
)
black_friday_dates = thanksgiving_dates + pd.Timedelta(days=1)
df["is_black_friday"] = df[DATE_COL].isin(black_friday_dates).astype(int)

# Christmas Eve / New Year's Eve
df["is_christmas_eve"] = (
    (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 24)
).astype(int)

df["is_new_year_eve"] = (
    (df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 31)
).astype(int)

# -----------------------------------------------------------------------------
# HOLIDAY PROXIMITY FEATURES
# -----------------------------------------------------------------------------
# "near holiday" windows often help more than holiday-day itself
for lag in [1, 2, 3, 7]:
    df[f"is_pre_holiday_{lag}"] = 0
    df[f"is_post_holiday_{lag}"] = 0

for h in holiday_dates:
    for lag in [1, 2, 3, 7]:
        df.loc[df[DATE_COL] == h - pd.Timedelta(days=lag), f"is_pre_holiday_{lag}"] = 1
        df.loc[df[DATE_COL] == h + pd.Timedelta(days=lag), f"is_post_holiday_{lag}"] = 1

# -----------------------------------------------------------------------------
# LONG WEEKEND / BRIDGE-DAY FEATURES
# -----------------------------------------------------------------------------
# Friday before holiday Monday, Monday after holiday weekend, etc.
df["is_monday"] = (df["day_of_week"] == 0).astype(int)
df["is_friday"] = (df["day_of_week"] == 4).astype(int)

df["is_bridge_day"] = 0
holiday_set = set(holiday_dates)

for i, d in enumerate(df[DATE_COL]):
    prev_day = d - pd.Timedelta(days=1)
    next_day = d + pd.Timedelta(days=1)
    # workday between holiday and weekend
    if (prev_day in holiday_set and d.dayofweek == 4) or (next_day in holiday_set and d.dayofweek == 0):
        df.at[i, "is_bridge_day"] = 1
# USE THE EXOGENOUS COLUMNS YOU CREATED
#EXOG_COLS = ["day_of_week", "doy", "moy", "is_thanksgiving", "is_christmas","is_weekend"]

EXOG_COLS = [
    # base
    "day_of_week", "day_of_month", "week_of_year", "week_of_month",
    "month", "quarter", "is_weekend",
    "is_month_start", "is_month_end", "is_quarter_start", "is_quarter_end",

    # special days of the week
    "is_monday", "is_friday",
    # holidays
    "is_holiday", "is_thanksgiving", "is_black_friday",
    "is_christmas", "is_christmas_eve", "is_new_year_eve",
    "is_pre_holiday_1", "is_pre_holiday_2", "is_pre_holiday_3", "is_pre_holiday_7",
    "is_post_holiday_1", "is_post_holiday_2", "is_post_holiday_3", "is_post_holiday_7",

    # boundary / behavior
    "is_bridge_day"
]



1082371


In [ ]:
df

,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,is_new_year_eve,is_pre_holiday_1,is_post_holiday_1,is_pre_holiday_2,is_post_holiday_2,is_pre_holiday_3,is_post_holiday_3,is_pre_holiday_7,is_post_holiday_7,is_bridge_day
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1,2021-01-23,55,6,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
2,2021-01-23,71,10,refrigerated baked gds,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
3,2021-01-23,116,16,dairy cream,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
4,2021-01-23,128,14,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082366,2023-02-22,983332,11,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1082367,2023-02-22,983754,4,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1082368,2023-02-22,988016,1,sparkling seltzer mixer,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1082369,2023-02-22,991921,8,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
print(df[TARGET_COL].describe())

count    1.082371e+06
mean     6.387899e+00
std      1.132080e+01
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      8.000000e+00
max      1.000000e+03
Name: value, dtype: float64


In [ ]:
# Filter for specific products if needed
target_products = [916110]
#target_products = [101054,101125, 101126, 102689, 103633, 103672, 103737, 103776, 103781,103782]
if target_products:
    products = df[df['item_id'].isin(target_products)][['item_id', 'store_id']].drop_duplicates().values
else:
    # Get all unique products from the subset dataset
    products = df[['item_id', 'store_id']].drop_duplicates().values

products = products[:5]  # Limit to first 5 products for testing
print(products)
# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)

In [ ]:
forecast_horizon = 152
seq_length = 30
train_size = 455
val_size = 154
lookback_window = 7 
BATCH_SIZE = 32
test_start_idx = len(df) - forecast_horizon
val_start_idx = test_start_idx - val_size
train_start_idx = val_start_idx - train_size
train_slice = slice(train_start_idx, val_start_idx)
val_slice = slice(val_start_idx, test_start_idx)
test_slice = slice(test_start_idx, None)
    
print(f"Train slice: {train_slice}, Val slice: {val_slice}, Test slice: {test_slice}")
# Extract Target
train = df[TARGET_COL][train_slice].values
val = df[TARGET_COL][val_slice].values
test = df[TARGET_COL][test_slice].values
# Scale Target
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()
test_scaled = scaler.transform(test.reshape(-1, 1)).flatten()

In [ ]:
# out_filename = f"dynamic_embeddings_{method}_Window{WINDOW_SIZE}_Step{STEP_SIZE}_p{p}.pkl"
GRAPH_WINDOW_SIZE = 15
STEP_SIZE = 1
p = 0.1 # percentile
method = "euclidean"

# Ensure the correct path to the embeddings is used
out_filename = f"dynamic_embeddings_{method}_Window{GRAPH_WINDOW_SIZE}_Step{STEP_SIZE}_p{p}.pkl"
out_path = os.path.join(f"../GraphAnalysis/DynamicGraphPkls/GraphEmbeddings/{method}/", out_filename)

with open(out_path, 'rb') as f:
    embedding_data = pickle.load(f)

# The dict contains: 'embeddings', 'node_mapping', 'method', 'percentile', 'dimensions'
all_embeddings = embedding_data['embeddings']
node_mapping = embedding_data['node_mapping']

# products is defined above as df[['item_id', 'store_id']].drop_duplicates().values
# Extract the specific node strings used in the mapping, often in the format 'item_id_store_id' or tuple format
# Check node_mapping keys format (assuming standard format "Item_ID" or "Item_ID_Store_ID", we'll adapt to how node2vec mapped them)
selected_product_nodes = []
for item, store in products:
    # Based on standard representation where nodes might be defined as string f"{item}_{store}" or similar.
    # We will grab the nodes matching the selected products
    node_str = f"{item}_{store}" 
    if node_str in node_mapping:
        selected_product_nodes.append(node_mapping[node_str])
    elif item in node_mapping: # Fallback if node is just item_id
        selected_product_nodes.append(node_mapping[item])
    elif (item, store) in node_mapping:
        selected_product_nodes.append(node_mapping[(item, store)])

# Slice the embeddings array for only the selected product indices
# all_embeddings shape: (num_days, num_nodes, emb_dim)
selected_embeddings = all_embeddings[:, selected_product_nodes, :]
print(f"Loaded embeddings with shape: {selected_embeddings.shape} for {len(selected_product_nodes)} products.")

Loading embeddings from GraphEmbeddings\dynamic_embeddings_euclidean_Window15_Step1_p0.1.pkl...


FileNotFoundError: [Errno 2] No such file or directory: 'GraphEmbeddings\\dynamic_embeddings_euclidean_Window15_Step1_p0.1.pkl'

In [ ]:
# Extract Exogenous Variables
exog_train = None
exog_val = None
exog_cols = EXOG_COLS
full_exog = None

if exog_cols and len(exog_cols) > 0:
    exog_train = df[exog_cols][train_slice].values
    exog_val = df[exog_cols][val_slice].values
    exog_test = df[exog_cols][test_slice].values
        # Scale Exogenous Variables
    exog_scaler = MinMaxScaler()
    exog_train_scaled = exog_scaler.fit_transform(exog_train)
    exog_val_scaled = exog_scaler.transform(exog_val)
    exog_test_scaled = exog_scaler.transform(exog_test)
else:
    exog_train_scaled = None
    exog_val_scaled = None
    exog_test_scaled = None
    # Input size = 1 (target) + number of exog features
input_size = 1 + (len(exog_cols) if exog_cols and len(exog_cols) > 0 else 0)

In [ ]:
  # -------------------------------------------------------------------------
    # 2. Datasets & Loaders
    # -------------------------------------------------------------------------
    # Pass exogenous data to TimeSeriesDataset
train_dataset = TimeSeriesDataset(train_scaled, exog_train_scaled, seq_length)
use_pin_memory = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=use_pin_memory)
    
# Important: Pass exog_val to validation dataset too!
val_dataset = TimeSeriesDataset(val_scaled, exog_val_scaled, seq_length)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=use_pin_memory)

In [ ]:
HIDDEN_SIZE = 32
NUM_LAYERS = 1
DROPOUT = 0.0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTM(input_size=input_size, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)


In [ ]:
LEARNING_RATE = 0.001
PATIENCE = 100
criterion = nn.MSELoss()
criterion2 = nn.MSELoss()  # Placeholder for potential multi-task loss

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=PATIENCE//3)

# Grid Search Cell

In [ ]:
import time
import itertools
import sys
import os
import importlib
from train import train_model
from test import test_model
from utils import compute_metrics
sys.path.append(os.path.abspath('..'))

# Reload plot_results to pick up changes in model_utils.utils
import model_utils.plots
importlib.reload(model_utils.plots)
from model_utils.plots import plot_results


# Modify lstm_experiment to return timings and handle plotting internally
def lstm_experiment_grid(df, target, item_id, store_id, train_size=500, val_size=100, forecast_window=161, 
                   seq_length=30, epochs=100, batch_size=32, lr=0.001, dropout=0.0, hidden_size=32, num_layers=1, exog_cols=None,
                   patience=50, seed=42, criterion=None, optimizer=None, save_plot_path=None, scheduler=False):

    model, train_losses, val_losses, best_epoch, train_time = train_model(
        df, target, item_id, store_id, train_size, val_size, forecast_window, seq_length, 
        epochs, batch_size, lr, dropout, hidden_size, num_layers, exog_cols,
        patience, seed, criterion, optimizer, save_plot_path, scheduler
    )
    
    forecast, inference_time = test_model(
        model, df, target, item_id, store_id, 
        train_size, val_size, forecast_window, seq_length)
    # Metrics
    rmse, mae, bias, score , pocid = compute_metrics(test, forecast)
    if save_plot_path:
        train_index = df[DATE_COL][train_slice].values
        val_index = df[DATE_COL][val_slice].values
        test_index = df[DATE_COL][test_slice].values
        
        plot_results(train, val, test, forecast, train_index, val_index, test_index, 
                     train_losses, val_losses, target, 
                     title=f'LSTM Forecast (Seed={seed}, Criterion={criterion}, Item={item_id}, Store={store_id})',
                     save_path=save_plot_path,
                     rmse=rmse, mae=mae, bias=bias, score=score, pocid=pocid)
    
    return rmse, mae, train_time, inference_time, best_epoch

In [ ]:
# Grid Search Parameters
seeds = [2024]
loss_functions = ['MSELoss']  
batch_size = 32
hidden_size = 32
num_layers = 1
dropout = 0.0
EPOCHS = 1000  # Ensure EPOCHS is defined
LEARNING_RATE = 0.001

results = []
os.makedirs('grid_search_plots', exist_ok=True)


[[916110   6269]]


In [ ]:
# check products series count and check for any missing dates
item_id, store_id = products[0]  # Just checking the first product for now
# Filter data for the specific product
df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
df_product

,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,is_new_year_eve,is_pre_holiday_1,is_post_holiday_1,is_pre_holiday_2,is_post_holiday_2,is_pre_holiday_3,is_post_holiday_3,is_pre_holiday_7,is_post_holiday_7,is_bridge_day
1202,2021-01-23,916110,60,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
2558,2021-01-24,916110,29,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
3931,2021-01-25,916110,5,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
5309,2021-01-26,916110,13,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
6696,2021-01-27,916110,31,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1077220,2023-02-18,916110,41,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1078594,2023-02-19,916110,35,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1079945,2023-02-20,916110,23,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1081234,2023-02-21,916110,25,eggs egg substitutes,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:


# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)

# Run Grid Search
print(f"Starting Grid Search with {len(seeds)} seeds, {len(loss_functions)} loss functions and {len(products)} products...")

for seed in seeds:
    print(f"\n--- Processing Seed: {seed} ---")
    for loss_type in loss_functions:
        print(f"\n--- Processing Loss Type: {loss_type} ---")
        
        for item_id, store_id in products:
            print(f"Running: Seed={seed}, Loss={loss_type}, Item={item_id}, Store={store_id}")
            
            # Filter data for the specific product
            df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
            
            # Handle DATE_COL (ensure it is a column and not in the index)
            if DATE_COL in df_product.index.names:
                if DATE_COL in df_product.columns:
                    # If it's in both, drop the index version to avoid "cannot insert" error
                    df_product = df_product.reset_index(drop=True)
                else:
                    # If it's only in the index, move it to a column
                    df_product = df_product.reset_index()

            # Fallback: simple reset to ensure RangeIndex 0..N
            df_product = df_product.reset_index(drop=True)

            df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
            df_product = df_product.sort_values(DATE_COL)
            df_product = df_product.reset_index(drop=True) # Final clean reset
            
            # Create directory for plots if it doesn't exist
            plot_dir = f'grid_search_plots/seed_{seed}/{loss_type}'
            os.makedirs(plot_dir, exist_ok=True)
            plot_filename = f'{plot_dir}/lstm_item{item_id}_store{store_id}.png'
            rmse, mae, train_time, infer_time, best_epoch = lstm_experiment_grid(
                df=df_product, 
                target=TARGET_COL, 
                item_id=item_id,
                store_id=store_id,
                train_size=train_size, 
                val_size=val_size,
                forecast_window=forecast_horizon, 
                seq_length=lookback_window,
                epochs=EPOCHS,  # Use the global EPOCHS setting (e.g., 1000)
                batch_size=batch_size, 
                lr=LEARNING_RATE,
                dropout=dropout,
                hidden_size=hidden_size,
                num_layers=1,
                patience=150,
                exog_cols=EXOG_COLS,
                seed=seed,
                loss_type=loss_type,
                save_plot_path=plot_filename,
                use_scheduler=True
            )
            
            results.append({
                'seed': seed,
                'loss_type': loss_type,
                'item_id': item_id,
                'store_id': store_id,
                'batch_size': batch_size,
                'hidden_size': hidden_size,
                'dropout': dropout,
                'rmse': rmse,
                'mae': mae,
                'train_time': train_time,
                'inference_time': infer_time,
                'best_epoch': best_epoch,
                'plot_path': plot_filename
            })
        

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.to_csv('grid_search_results.csv', index=False)

Starting Grid Search with 1 seeds, 1 loss functions and 1 products...

--- Processing Seed: 2024 ---

--- Processing Loss Type: MSELoss ---
Running: Seed=2024, Loss=MSELoss, Item=916110, Store=6269
Calculated indices - Train Start: 0, Val Start: 455, Test Start: 609
Epoch 10/1000 | Train Loss: 0.020006 | Val Loss: 0.020183
Epoch 20/1000 | Train Loss: 0.017677 | Val Loss: 0.019472
Epoch 30/1000 | Train Loss: 0.017082 | Val Loss: 0.019433
Epoch 40/1000 | Train Loss: 0.016518 | Val Loss: 0.019184
Epoch 50/1000 | Train Loss: 0.016033 | Val Loss: 0.018960
Epoch 60/1000 | Train Loss: 0.015631 | Val Loss: 0.018775
Epoch 70/1000 | Train Loss: 0.015247 | Val Loss: 0.018617
Epoch 80/1000 | Train Loss: 0.014874 | Val Loss: 0.018511
Epoch 90/1000 | Train Loss: 0.014531 | Val Loss: 0.018477
Epoch 100/1000 | Train Loss: 0.014219 | Val Loss: 0.018505
Epoch 110/1000 | Train Loss: 0.013924 | Val Loss: 0.018578
Epoch 120/1000 | Train Loss: 0.013621 | Val Loss: 0.018693
Epoch 130/1000 | Train Loss: 0.013

In [ ]:
#df_inference= pd.read_csv('inference_logs/seed_2024/MSELoss/inference_item916110_store6269.csv')
#df_inference